# 2. Nettoyage et Préprocessing des Avis d'Assurance

Ce notebook nettoie et prépare les données pour les analyses suivantes.

In [34]:
import re
import warnings

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from spellchecker import SpellChecker
from tqdm.auto import tqdm

from config import DATA_CLEAN, TEXT_COL, RATING_COL, LENGTH_COL, CLEAN_COL

In [35]:
warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## 2.1 Chargement des données

In [36]:
df = pd.read_csv(f"{BASE_DIR}/data/processed/data.csv")

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34435 entries, 0 to 34434
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   note              24104 non-null  float64
 1   auteur            34434 non-null  str    
 2   avis              34435 non-null  str    
 3   assureur          34435 non-null  str    
 4   produit           34435 non-null  str    
 5   type              34435 non-null  str    
 6   date_publication  34435 non-null  str    
 7   date_exp          34435 non-null  str    
 8   avis_en           34433 non-null  str    
 9   avis_cor          435 non-null    str    
 10  avis_cor_en       431 non-null    str    
dtypes: float64(1), str(10)
memory usage: 27.0 MB


## 2.2 Suppression des doublons et valeurs manquantes

In [38]:
nb_ligne_before = len(df)
nb_ligne_before

34435

In [39]:
df = df.drop_duplicates()

In [40]:
len(df)

34430

In [41]:
df = df.dropna(subset=[RATING_COL])

In [42]:
len(df)

24100

## 2.3 Normalisation du texte

In [43]:
def normalize_text(text):
    """Normalisation de base pour le français"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = re.sub(r"[^a-zàâäéèêëîïôöùûüçœæ\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [44]:
sample = "J'ai eu une MAUVAISE expérience avec cette assurance ! Ils ont refusé mon dossier. www.assurance.fr"
print("Original :", sample)
print("Normalisé :", normalize_text(sample))

Original : J'ai eu une MAUVAISE expérience avec cette assurance ! Ils ont refusé mon dossier. www.assurance.fr
Normalisé : j'ai eu une mauvaise expérience avec cette assurance ils ont refusé mon dossier


## 2.4 Suppression des stopwords

In [45]:
keep_words = {'ne', 'pas', 'jamais', 'rien', 'personne', 'aucun', 'aucune', 'sans', 'ni', 'non'}
stopwords_fr = set(stopwords.words('french')) - keep_words

In [46]:
def remove_stopwords(text):
    """Supprime les stopwords français."""
    tokens = text.split()
    return ' '.join([w for w in tokens if w not in stopwords_fr])

In [47]:
sample_clean = normalize_text(sample)
print("Avec stopwords    :", sample_clean)
print("Sans stopwords    :", remove_stopwords(sample_clean))

Avec stopwords    : j'ai eu une mauvaise expérience avec cette assurance ils ont refusé mon dossier
Sans stopwords    : j'ai mauvaise expérience cette assurance refusé dossier


## 2.5 Correction orthographique

In [48]:
spell = SpellChecker(language='fr')

In [49]:
def correct_spelling(text):
    """Correction orthographique française."""
    return ' '.join(
        [spell.correction(word) or word if len(word) > 2 and word.isalpha() else word for word in text.split()])

In [50]:
test_txt = "l'assurence a refusé mon dossié sans explication"
print("Original  :", test_txt)
print("Corrigé   :", correct_spelling(test_txt))

Original  : l'assurence a refusé mon dossié sans explication
Corrigé   : l'assurence a refusé mon dossier sans explication


## 2.6 Lemmatisation avec spaCy

In [51]:
nlp = spacy.load('fr_core_news_sm', disable=['parser', 'ner'])

In [52]:
def lemmatize_text(text):
    """Lemmatise le texte français avec spaCy, en conservant les mots de négation."""
    doc = nlp(str(text)[:500])
    return ' '.join([
        token.lemma_ for token in doc
        if (not token.is_stop or token.text in keep_words) and not token.is_punct and len(token.text) > 2
    ])

In [53]:
test_sentence = "l'assurance a refusé de rembourser mes frais médicaux sans aucune explication"
print("Original  :", test_sentence)
print("Lemmatisé :", lemmatize_text(test_sentence))

Original  : l'assurance a refusé de rembourser mes frais médicaux sans aucune explication
Lemmatisé : assurance refuser rembourser frais médical sans aucun explication


## 2.7 Pipeline de preprocessing complet

In [54]:
def preprocess_pipeline(text):
    """Pipeline complet de préprocessing pour le français."""
    if pd.isna(text) or str(text).strip() == '':
        return ''

    text = normalize_text(text)
    text = correct_spelling(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)

    return text

In [55]:
test_reviews = [
    "J'ai eu une terrible expérience avec cette compagnie d'assurance ! Ils ont refusé mon dossier sans raison.",
    "Très bonne couverture et prix compétitifs. Le service client a été très réactif.",
    "Résiliation de mon contrat après 3 mois. La prime a augmenté sans préavis."
]

print("Test du pipeline\n")
for review in test_reviews:
    cleaned = preprocess_pipeline(review)
    print(f"Original : {review[:100]}")
    print(f"Nettoyé  : {cleaned[:100]}")
    print()

Test du pipeline

Original : J'ai eu une terrible expérience avec cette compagnie d'assurance ! Ils ont refusé mon dossier sans r
Nettoyé  : terrible expérience compagnie assurance refuser dossier sans raison

Original : Très bonne couverture et prix compétitifs. Le service client a été très réactif.
Nettoyé  : bon couvertur prix compétitif service client réactif

Original : Résiliation de mon contrat après 3 mois. La prime a augmenté sans préavis.
Nettoyé  : résiliation contrat mois prime augmenter sans préavis



In [56]:
tqdm.pandas(desc="application du pipeline", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")

df[CLEAN_COL] = df[TEXT_COL].progress_apply(preprocess_pipeline)
print(f"Preprocessing terminé sur {len(df)} avis !")

df[LENGTH_COL] = df[TEXT_COL].str.len()
df['avis_clean_len'] = df[CLEAN_COL].str.len()
df['nb_mots'] = df[CLEAN_COL].str.split().str.len()

print(f"Longueur moyenne originale : {df[LENGTH_COL].mean():.0f} car.")
print(f"Longueur moyenne nettoyée  : {df['avis_clean_len'].mean():.0f} car.")
print(f"Mots moyens (nettoyé)      : {df['nb_mots'].mean():.1f}")

application du pipeline:   0%|          | 120/24100 [00:08<29:39]


KeyboardInterrupt: 

## 2.8 Analyse post-nettoyage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

vectorizer_uni = CountVectorizer(max_features=30, ngram_range=(1, 1))
X_uni = vectorizer_uni.fit_transform(df[CLEAN_COL].fillna(''))
word_freq = pd.DataFrame({
    'word': vectorizer_uni.get_feature_names_out(),
    'count': X_uni.toarray().sum(axis=0)
}).sort_values('count', ascending=True)

axes[0].barh(word_freq['word'].tail(20), word_freq['count'].tail(20))
axes[0].set_title('Top 20 mots (après nettoyage)')
axes[0].set_xlabel('Fréquence')

# Bigrammes
vectorizer_bi = CountVectorizer(max_features=20, ngram_range=(2, 2))
X_bi = vectorizer_bi.fit_transform(df[CLEAN_COL].fillna(''))
bigram_freq = pd.DataFrame({
    'bigram': vectorizer_bi.get_feature_names_out(),
    'count': X_bi.toarray().sum(axis=0)
}).sort_values('count', ascending=True)

axes[1].barh(bigram_freq['bigram'].tail(15), bigram_freq['count'].tail(15))
axes[1].set_title('Top 15 bigrammes (après nettoyage)')
axes[1].set_xlabel('Fréquence')

plt.tight_layout()
plt.show()

## 2.9 Export des données nettoyées

In [ ]:
output_path = DATA_CLEAN
df.to_csv(output_path, index=False)
print(f"Dataset nettoyé sauvegardé")

In [ ]:
df.head()